In [1]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "timeSeries").exists() and ROOT.name == "timeSeries":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "timeSeries" / "dataset"
RESULT_DIR = ROOT / "timeSeries" / "results" / "model_2_rolling"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["pickup_hour"])
val = pd.read_csv(DATA_DIR / "val.csv", parse_dates=["pickup_hour"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["pickup_hour"])

for df in (train, val, test):
    df["trip_count"] = pd.to_numeric(df["trip_count"], errors="coerce")
    df.dropna(subset=["PULocationID", "pickup_hour", "trip_count"], inplace=True)

train = train.sort_values(["PULocationID", "pickup_hour"])
val = val.sort_values(["PULocationID", "pickup_hour"])
test = test.sort_values(["PULocationID", "pickup_hour"])

assert train["pickup_hour"].max() < val["pickup_hour"].min()
assert val["pickup_hour"].max() < test["pickup_hour"].min()

HORIZON = 168  # Forecast one week at a time
MIN_TRAIN_POINTS = 24 * 14

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Train: (403374, 17)
Validation: (86388, 17)
Test: (86526, 17)


In [2]:
candidate_file = (
    ROOT / "timeSeries" / "results" / "model_2"
    / "validation_candidate_summary.csv"
)

if not candidate_file.exists():
    raise FileNotFoundError(
        f"Cannot find {candidate_file}. "
        "Run Model 2 first and ensure its validation results were saved."
    )

candidates = pd.read_csv(candidate_file)
best_label = candidates.sort_values("MAE").iloc[0]["model"]

match = re.fullmatch(
    r"DHR_D(\d+)_W(\d+)_ARIMA\((\d+), (\d+), (\d+)\)",
    str(best_label)
)

if not match:
    raise ValueError(f"Unexpected model configuration: {best_label}")

DAILY_K = int(match.group(1))
WEEKLY_K = int(match.group(2))
ARIMA_ORDER = tuple(int(match.group(i)) for i in (3, 4, 5))

print("Selected model:", best_label)
print("Daily Fourier harmonics:", DAILY_K)
print("Weekly Fourier harmonics:", WEEKLY_K)
print("ARIMA order:", ARIMA_ORDER)

Selected model: DHR_D3_W3_ARIMA(1, 0, 1)
Daily Fourier harmonics: 3
Weekly Fourier harmonics: 3
ARIMA order: (1, 0, 1)


In [3]:
def make_fourier(index, daily_k, weekly_k):
    index = pd.DatetimeIndex(index)
    t = (
        (index - pd.Timestamp("1970-01-01"))
        / pd.Timedelta(hours=1)
    ).to_numpy(dtype=float)

    features = {}

    for period, kmax, prefix in [
        (24.0, daily_k, "daily"),
        (168.0, weekly_k, "weekly")
    ]:
        for k in range(1, kmax + 1):
            features[f"{prefix}_sin_{k}"] = np.sin(
                2 * np.pi * k * t / period
            )
            features[f"{prefix}_cos_{k}"] = np.cos(
                2 * np.pi * k * t / period
            )

    return pd.DataFrame(features, index=index)


def get_zone_series(frame, zone, end_time):
    zone_data = frame.loc[
        (frame["PULocationID"] == zone)
        & (frame["pickup_hour"] <= end_time),
        ["pickup_hour", "trip_count"]
    ]

    series = (
        zone_data.groupby("pickup_hour")["trip_count"]
        .sum()
        .sort_index()
    )

    if series.empty:
        return pd.Series(dtype=float)

    hourly_index = pd.date_range(
        series.index.min(), series.index.max(), freq="h"
    )

    series = series.reindex(hourly_index).fillna(0.0).astype(float)
    series.index.name = "pickup_hour"

    return series


def forecast_dhr(history, future_index):
    if len(history) < MIN_TRAIN_POINTS:
        raise ValueError("Insufficient historical observations")

    x_train = make_fourier(history.index, DAILY_K, WEEKLY_K)
    x_future = make_fourier(future_index, DAILY_K, WEEKLY_K)

    model = SARIMAX(
        history.to_numpy(dtype=float),
        exog=x_train.to_numpy(),
        order=ARIMA_ORDER,
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted = model.fit(disp=False, maxiter=100)

    forecast = fitted.get_forecast(
        steps=len(future_index),
        exog=x_future.to_numpy()
    ).predicted_mean

    return np.maximum(np.asarray(forecast, dtype=float), 0.0)

In [4]:
def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual, predicted = actual[valid], predicted[valid]

    nonzero = actual != 0

    return {
        "n_rows": len(actual),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "MAPE_pct": (
            np.mean(
                np.abs(
                    (actual[nonzero] - predicted[nonzero])
                    / actual[nonzero]
                )
            ) * 100
            if nonzero.any() else np.nan
        ),
        "mape_excluded_zero_count": int((~nonzero).sum()),
        "mape_excluded_zero_pct": float((~nonzero).mean() * 100)
    }

In [5]:
# History available before the first test forecast.
history_data = pd.concat([train, val], ignore_index=True)

test_start = test["pickup_hour"].min()
test_end = test["pickup_hour"].max()

origins = pd.date_range(
    start=test_start,
    end=test_end,
    freq=f"{HORIZON}h"
)

zones = sorted(test["PULocationID"].unique())

all_predictions = []
fold_results = []
failure_rows = []

for fold_number, origin in enumerate(origins, start=1):
    fold_end = min(
        origin + pd.Timedelta(hours=HORIZON - 1),
        test_end
    )

    future_index = pd.date_range(origin, fold_end, freq="h")

    print(
        f"Fold {fold_number}: "
        f"{origin} -> {fold_end}"
    )

    fold_predictions = []

    for zone in zones:
        actual_zone = test.loc[
            (test["PULocationID"] == zone)
            & (test["pickup_hour"] >= origin)
            & (test["pickup_hour"] <= fold_end),
            ["PULocationID", "pickup_hour", "trip_count"]
        ]

        if actual_zone.empty:
            continue

        history = get_zone_series(
            history_data, zone, origin - pd.Timedelta(hours=1)
        )

        try:
            forecast = forecast_dhr(history, future_index)
            forecast_series = pd.Series(
                forecast, index=future_index
            )

            for row in actual_zone.itertuples(index=False):
                fold_predictions.append({
                    "fold": fold_number,
                    "PULocationID": zone,
                    "pickup_hour": row.pickup_hour,
                    "actual": row.trip_count,
                    "predicted": float(
                        forecast_series.loc[row.pickup_hour]
                    )
                })

        except Exception as exc:
            failure_rows.append({
                "fold": fold_number,
                "PULocationID": zone,
                "error": f"{type(exc).__name__}: {exc}"
            })

    fold_df = pd.DataFrame(fold_predictions)

    if not fold_df.empty:
        fold_results.append({
            "fold": fold_number,
            "forecast_start": origin,
            "forecast_end": fold_end,
            **calculate_metrics(
                fold_df["actual"], fold_df["predicted"]
            )
        })

        all_predictions.extend(fold_predictions)

    # Actual values from this completed window become available
    # only after its forecasts have been generated.
    history_data = pd.concat(
        [
            history_data,
            test.loc[
                (test["pickup_hour"] >= origin)
                & (test["pickup_hour"] <= fold_end)
            ]
        ],
        ignore_index=True
    )

fold_metrics = pd.DataFrame(fold_results)
predictions_df = pd.DataFrame(all_predictions)

fold_metrics.to_csv(
    RESULT_DIR / "rolling_origin_metrics.csv", index=False
)
predictions_df.to_csv(
    RESULT_DIR / "rolling_origin_predictions.csv", index=False
)
pd.DataFrame(failure_rows).to_csv(
    RESULT_DIR / "rolling_origin_failures.csv", index=False
)

display(fold_metrics)

print("Saved results to:", RESULT_DIR)

Fold 1: 2026-02-02 21:00:00 -> 2026-02-09 20:00:00
Fold 2: 2026-02-09 21:00:00 -> 2026-02-16 20:00:00
Fold 3: 2026-02-16 21:00:00 -> 2026-02-23 20:00:00
Fold 4: 2026-02-23 21:00:00 -> 2026-02-28 23:00:00


,fold,forecast_start,forecast_end,n_rows,MAE,RMSE,R2,MAPE_pct,mape_excluded_zero_count,mape_excluded_zero_pct
0,1,2026-02-02 21:00:00,2026-02-09 20:00:00,23184,10.656963,24.293852,0.887284,64.157810,6266,27.027260
1,2,2026-02-09 21:00:00,2026-02-16 20:00:00,23184,10.838727,25.149026,0.871279,72.319651,6447,27.807971
2,3,2026-02-16 21:00:00,2026-02-23 20:00:00,23184,13.647745,34.603668,0.714604,133.497181,6697,28.886301
3,4,2026-02-23 21:00:00,2026-02-28 23:00:00,16974,11.157452,24.525970,0.879846,84.438086,4603,27.117945


Saved results to: /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/timeSeries/results/model_2_rolling


In [6]:
if predictions_df.empty:
    raise RuntimeError(
        "No forecasts were generated. Check the model failures file."
    )

overall_metrics = pd.DataFrame([{
    "model": "Dynamic Harmonic Regression",
    **calculate_metrics(
        predictions_df["actual"],
        predictions_df["predicted"]
    )
}])

overall_metrics.to_csv(
    RESULT_DIR / "rolling_origin_overall_metrics.csv",
    index=False
)

display(overall_metrics)

,model,n_rows,MAE,RMSE,R2,MAPE_pct,mape_excluded_zero_count,mape_excluded_zero_pct
0,Dynamic Harmonic Regression,86526,11.605205,27.68473,0.841443,88.643761,24013,27.752352
